In [1]:
## import libraries

import pandas as pd
import numpy as np

In [2]:
## load the dataset
df = pd.read_csv("data/ad_performance.csv")
df.head()

,date,ad_id,impressions,clicks,spend,conversions,frequency,behavior
0,2026-08-01,AD_001,6031,179,90.89,7,1.18,stable
1,2026-08-02,AD_001,9010,325,151.87,19,1.07,stable
2,2026-08-03,AD_001,6153,198,91.87,9,1.20,stable
3,2026-08-04,AD_001,4590,135,83.12,14,1.25,stable
4,2026-08-05,AD_001,5875,191,84.73,9,1.17,stable


In [3]:
## Check data structure
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         600 non-null    object 
 1   ad_id        600 non-null    object 
 2   impressions  600 non-null    int64  
 3   clicks       600 non-null    int64  
 4   spend        600 non-null    float64
 5   conversions  600 non-null    int64  
 6   frequency    600 non-null    float64
 7   behavior     600 non-null    object 
dtypes: float64(2), int64(3), object(3)
memory usage: 37.6+ KB


In [4]:
df["date"] = pd.to_datetime(df["date"])


In [5]:
## check data quality
print("Missing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nDataset shape:")
print(df.shape)

Missing values:
date           0
ad_id          0
impressions    0
clicks         0
spend          0
conversions    0
frequency      0
behavior       0
dtype: int64

Duplicate rows:
0

Dataset shape:
(600, 8)


In [6]:
## Feature Engineering
df["ctr"] = (df["clicks"] / df["impressions"])*100
df["cpc"] = (df["spend"] / df["clicks"].replace(0, np.nan))
df["cpm"] = (df["spend"] / df["impressions"]) * 1000
df["cpa"] = (df["spend"] / df["conversions"].replace(0, np.nan))

df[
    [
        "ad_id",
        "date",
        "ctr",
        "cpc",
        "cpm",
        "cpa",
        "frequency"
    ]
].head(10)

,ad_id,date,ctr,cpc,cpm,cpa,frequency
0,AD_001,2026-08-01,2.967999,0.507765,15.070469,12.984286,1.18
1,AD_001,2026-08-02,3.607103,0.467292,16.855716,7.993158,1.07
2,AD_001,2026-08-03,3.217942,0.463990,14.930928,10.207778,1.20
3,AD_001,2026-08-04,2.941176,0.615704,18.108932,5.937143,1.25
4,AD_001,2026-08-05,3.251064,0.443613,14.422128,9.414444,1.17
5,AD_001,2026-08-06,3.428030,0.470608,16.132576,6.084286,1.33
6,AD_001,2026-08-07,2.844856,0.572965,16.300033,24.637500,1.32
7,AD_001,2026-08-08,3.243368,0.466214,15.121047,10.878333,1.37
8,AD_001,2026-08-09,2.764522,0.594762,16.442324,19.984000,1.46
9,AD_001,2026-08-10,3.164887,0.444239,14.059651,11.994444,1.38


In [7]:
df[["ctr", "cpc", "cpm", "cpa", "frequency"]].describe()

,ctr,cpc,cpm,cpa,frequency
count,600.000000,600.000000,600.000000,592.000000,600.000000
mean,2.373169,0.803045,16.040413,16.227835,2.135967
std,0.793786,0.536971,5.491712,17.267192,0.612494
min,0.543150,0.222011,6.923542,2.331111,1.000000
25%,1.733264,0.480612,10.580899,7.726250,1.627500
50%,2.522176,0.637225,16.140438,11.554968,2.150000
75%,2.966331,0.904772,19.624200,17.528864,2.662500
max,4.346098,3.236286,36.056024,153.290000,3.290000


In [8]:
import plotly.express as px
ad_1 = df[df["ad_id"] == "AD_001"]

fig = px.line(
    ad_1,
    x="date",
    y="ctr",
    title="CTR Over Time — AD_001"
)

fig.show()

In [9]:
fatigue_id = df[df["ad_id"]=="AD_005"]

fig =px.line(
    fatigue_id,
    x="date",
    y="ctr",
    title="CTR Over Time — Gradual Deterioration"

)

fig.show()

In [10]:
## Comparing both together
comparison = df[
    df["ad_id"].isin(["AD_001", "AD_005"])
]

fig = px.line(
    comparison,
    x="date",
    y="ctr",
    color="ad_id",
    title="Stable vs Deteriorating Ad Performance"
)

fig.show()

In [11]:
## saving processed data
df.to_csv("data/processed_ad_performance.csv", index=False)

In [12]:
import ruptures as rpt
ad_8 = df[df["ad_id"] == "AD_008"].copy()

ad_8 = ad_8.sort_values("date")

ad_8.head()


,date,ad_id,impressions,clicks,spend,conversions,frequency,behavior,ctr,cpc,cpm,cpa
420,2026-08-01,AD_008,5453,106,81.26,9,1.00,sudden_fatigue,1.943884,0.766604,14.901889,9.028889
421,2026-08-02,AD_008,8196,166,120.63,8,1.03,sudden_fatigue,2.025378,0.726687,14.718155,15.078750
422,2026-08-03,AD_008,4049,89,59.21,5,1.12,sudden_fatigue,2.198074,0.665281,14.623364,11.842000
423,2026-08-04,AD_008,9897,195,129.71,11,1.16,sudden_fatigue,1.970294,0.665179,13.105992,11.791818
424,2026-08-05,AD_008,5202,139,75.20,10,1.19,sudden_fatigue,2.672049,0.541007,14.455978,7.520000


In [13]:
ctr_values = ad_8["ctr"].to_numpy()

print(ctr_values[:10])
print(ctr_values.shape)

[1.9438841  2.02537823 2.1980736  1.97029403 2.67204921 1.90742964
 1.99628598 1.77252585 1.98639456 2.06980073]
(60,)


In [14]:
model = rpt.Pelt(
    model="l2",
    min_size=5
)

model.fit(ctr_values)

change_points = model.predict(pen=1)

print("Detected change points:", change_points)

Detected change points: [35, 60]


In [15]:
change_points = [35, 60]

actual_changes = change_points[:-1]

print(actual_changes)

[35]


In [16]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=ad_8["date"],
        y=ad_8["ctr"],
        mode="lines",
        name="CTR"
    )
)

for point in change_points[:-1]:

    fig.add_vline(
        x=ad_8["date"].iloc[point],
        line_dash="dash",
        line_color="red"
    )

fig.update_layout(
    title="AD_008 — CTR Change-Point Detection",
    xaxis_title="Date",
    yaxis_title="CTR (%)"
)

fig.show()

In [17]:
truth = pd.read_csv("data/ground_truth.csv")

truth[truth["ad_id"] == "AD_008"]

,ad_id,behavior,fatigue_start_day
7,AD_008,sudden_fatigue,35.0


In [18]:
actual_start = 35

detected_points = change_points[:-1]

if detected_points:

    closest_point = min(
        detected_points,
        key=lambda x: abs(x - actual_start)
    )

    error = abs(closest_point - actual_start)

    print("Actual change index:", actual_start)
    print("Closest detected index:", closest_point)
    print("Absolute error:", error, "days")

else:
    print("No change point detected.")

Actual change index: 35
Closest detected index: 35
Absolute error: 0 days


In [19]:
ad_5 = df[df["ad_id"] == "AD_005"].copy()

ad_5 = ad_5.sort_values("date")

ctr_values_5 = ad_5["ctr"].to_numpy()

model_5 = rpt.Pelt(model="l2", min_size=5)

model_5.fit(ctr_values_5)

change_points_5 = model_5.predict(pen=1)

print(change_points_5)

[40, 55, 60]


In [20]:
actual_changes_5 = change_points_5[:-1]

print(actual_changes_5)

[40, 55]


In [21]:
import plotly.graph_objects as go

fig = go.Figure()

# Plot AD_005's CTR
fig.add_trace(
    go.Scatter(
        x=ad_5["date"],
        y=ad_5["ctr"],
        mode="lines",
        name="CTR"
    )
)

# Add vertical lines for detected change points
for point in actual_changes_5:

    fig.add_vline(
        x=ad_5["date"].iloc[point],
        line_dash="dash",
        line_color="red"
    )

# Add graph labels
fig.update_layout(
    title="AD_005 — Gradual Deterioration Detection",
    xaxis_title="Date",
    yaxis_title="CTR (%)"
)

fig.show()

In [22]:
# Select the stable advertisement
ad_1 = df[df["ad_id"] == "AD_001"].copy()

# Arrange observations by date
ad_1 = ad_1.sort_values("date")

# Extract CTR values
ctr_values_1 = ad_1["ctr"].to_numpy()

# Create the PELT model
model_1 = rpt.Pelt(model="l2", min_size=5)

# Analyze the CTR values
model_1.fit(ctr_values_1)

# Detect change points
change_points_1 = model_1.predict(pen=1)

# Remove the final endpoint
actual_changes_1 = change_points_1[:-1]

print("AD_001 detected changes:", actual_changes_1)

AD_001 detected changes: []


In [23]:
## Making a function so that all advertisments can be analyzed for change point detection
def detect_changes(ad_data):

    ad_data = ad_data.sort_values("date")

    ctr_values = ad_data["ctr"].to_numpy()

    model = rpt.Pelt(model="l2", min_size=5)

    model.fit(ctr_values)

    change_points = model.predict(pen=1)

    return change_points[:-1]

In [24]:
results = {}

for ad_id, ad_data in df.groupby("ad_id"):

    changes = detect_changes(ad_data)

    results[ad_id] = changes

print(results)

{'AD_001': [], 'AD_002': [], 'AD_003': [], 'AD_004': [], 'AD_005': [40, 55], 'AD_006': [40, 50], 'AD_007': [40], 'AD_008': [35], 'AD_009': [35], 'AD_010': [35]}


In [25]:
change_results = pd.DataFrame(
    [
        {
            "ad_id": ad_id,
            "change_points": changes,
            "num_changes": len(changes)
        }
        for ad_id, changes in results.items()
    ]
)

change_results

,ad_id,change_points,num_changes
0,AD_001,[],0
1,AD_002,[],0
2,AD_003,[],0
3,AD_004,[],0
4,AD_005,"[40, 55]",2
5,AD_006,"[40, 50]",2
6,AD_007,[40],1
7,AD_008,[35],1
8,AD_009,[35],1
9,AD_010,[35],1


In [26]:
features = ["ctr", "cpc", "frequency"]
X = df[features].copy()
print(X.head())
print(X.shape)

        ctr       cpc  frequency
0  2.967999  0.507765       1.18
1  3.607103  0.467292       1.07
2  3.217942  0.463990       1.20
3  2.941176  0.615704       1.25
4  3.251064  0.443613       1.17
(600, 3)


In [27]:
print(X.isnull().sum())
df["cpc"] = df["spend"] / df["clicks"].replace(0, np.nan)


ctr          0
cpc          0
frequency    0
dtype: int64


In [28]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X)

print(X_scaled[:5])
print(X_scaled.shape)

[[ 0.74998239 -0.55035813 -1.56208069]
 [ 1.55578845 -0.62579393 -1.74182428]
 [ 1.06512054 -0.63194914 -1.52940004]
 [ 0.71616398 -0.349177   -1.44769841]
 [ 1.10688115 -0.66992948 -1.57842102]]
(600, 3)


In [29]:
from sklearn.ensemble import IsolationForest

iso_model = IsolationForest(
    n_estimators=100,
    contamination=0.1,
    random_state=42
)

iso_model.fit(X_scaled)

,n_estimators,100
,max_samples,'auto'
,contamination,0.1
,max_features,1.0
,bootstrap,False
,n_jobs,None
,random_state,42
,verbose,0
,warm_start,False


In [30]:
predictions = iso_model.predict(X_scaled)

print(np.unique(predictions, return_counts=True))

(array([-1,  1]), array([ 60, 540]))


In [31]:
df["anomaly"] = predictions

df["anomaly_score"] = -iso_model.score_samples(X_scaled)

df[["ad_id", "date", "ctr", "cpc", "frequency", "anomaly", "anomaly_score"]].head()

,ad_id,date,ctr,cpc,frequency,anomaly,anomaly_score
0,AD_001,2026-08-01,2.967999,0.507765,1.18,1,0.477358
1,AD_001,2026-08-02,3.607103,0.467292,1.07,-1,0.570587
2,AD_001,2026-08-03,3.217942,0.463990,1.20,1,0.483245
3,AD_001,2026-08-04,2.941176,0.615704,1.25,1,0.452232
4,AD_001,2026-08-05,3.251064,0.443613,1.17,1,0.497063


In [32]:
anomaly_results = (
    df.groupby("ad_id")["anomaly"]
    .apply(lambda x: (x == -1).sum())
    .reset_index(name="num_anomalies")
)

anomaly_results

,ad_id,num_anomalies
0,AD_001,1
1,AD_002,7
2,AD_003,5
3,AD_004,0
4,AD_005,3
5,AD_006,3
6,AD_007,10
7,AD_008,8
8,AD_009,1
9,AD_010,22


In [33]:
def calculate_ctr_decline(ad_data):

    ad_data = ad_data.sort_values("date")

    first_ctr = ad_data["ctr"].iloc[:15].mean()

    last_ctr = ad_data["ctr"].iloc[-15:].mean()

    if first_ctr <= 0:
        return 0

    decline = ((first_ctr - last_ctr) / first_ctr) * 100

    return max(0, decline)

In [34]:
def calculate_cpc_increase(ad_data):

    ad_data = ad_data.sort_values("date")

    first_cpc = ad_data["cpc"].iloc[:15].mean()

    last_cpc = ad_data["cpc"].iloc[-15:].mean()

    if pd.isna(first_cpc) or first_cpc <= 0:
        return 0

    if pd.isna(last_cpc):
        return 0

    increase = ((last_cpc - first_cpc) / first_cpc) * 100

    return max(0, increase)

In [35]:
def calculate_fatigue_score(ad_data, change_points):

    ctr_decline = calculate_ctr_decline(ad_data)

    cpc_increase = calculate_cpc_increase(ad_data)

    anomaly_count = (ad_data["anomaly"] == -1).sum()

    anomaly_rate = anomaly_count / len(ad_data)

    ctr_score = min(40, ctr_decline * 0.8)

    cpc_score = min(20, cpc_increase * 0.2)

    change_score = 20 if len(change_points) > 0 else 0

    anomaly_score = min(20, anomaly_rate * 100)

    total_score = (
        ctr_score
        + cpc_score
        + change_score
        + anomaly_score
    )

    return round(total_score, 2)

In [39]:
ad_5 = df[df["ad_id"] == "AD_005"].copy()

ad_5 = ad_5.sort_values("date")

print(ad_5.columns.tolist())

['date', 'ad_id', 'impressions', 'clicks', 'spend', 'conversions', 'frequency', 'behavior', 'ctr', 'cpc', 'cpm', 'cpa', 'anomaly', 'anomaly_score']


In [40]:
score = calculate_fatigue_score(
    ad_5,
    results["AD_005"]
)

print("AD_005 Fatigue Score:", score) 

AD_005 Fatigue Score: 79.7


In [41]:
fatigue_results = []

for ad_id, ad_data in df.groupby("ad_id"):

    changes = results[ad_id]

    score = calculate_fatigue_score(ad_data, changes)

    if score >= 70:
        risk_level = "High"

    elif score >= 40:
        risk_level = "Medium"

    else:
        risk_level = "Low"

    fatigue_results.append({
        "ad_id": ad_id,
        "fatigue_score": score,
        "risk_level": risk_level
    })

fatigue_df = pd.DataFrame(fatigue_results)

fatigue_df

,ad_id,fatigue_score,risk_level
0,AD_001,8.89,Low
1,AD_002,11.67,Low
2,AD_003,9.29,Low
3,AD_004,2.82,Low
4,AD_005,79.70,High
5,AD_006,82.19,High
6,AD_007,96.67,High
7,AD_008,93.33,High
8,AD_009,81.67,High
9,AD_010,95.95,High


In [42]:
fatigue_df = fatigue_df.sort_values(
    "fatigue_score",
    ascending=False
)

fatigue_df

,ad_id,fatigue_score,risk_level
6,AD_007,96.67,High
9,AD_010,95.95,High
7,AD_008,93.33,High
5,AD_006,82.19,High
8,AD_009,81.67,High
4,AD_005,79.70,High
1,AD_002,11.67,Low
2,AD_003,9.29,Low
0,AD_001,8.89,Low
3,AD_004,2.82,Low
